# Stress Prediction V29 — Near-Full Seed Bagging
동일 계산량의 단일 seed 1,200개 트리와 3개 seed × 400개 트리를 Near-Full 검증에서 비교합니다.

Pooled 분위수·seed별 분위수 평균·중앙값 집계를 사전 고정하여 선택 과적합을 줄입니다.

In [1]:
from __future__ import annotations

import argparse
import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold


TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.52
PAIR_QUANTILE = 0.52
PAIR_WEIGHT = 0.15

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    """다른 행이나 Test 전체 통계를 사용하지 않는 행 단위 파생변수."""
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (
        result["glucose"] + 1e-6
    )
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, additional_copies in FEATURE_COPY_COUNTS.items():
        for copy_index in range(1, additional_copies + 1):
            result[f"{feature}__copy{copy_index}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant", fill_value="missing"
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore", sparse_output=False
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(strategy="median", add_indicator=True),
                numerical,
            ),
        ]
    )


def tree_quantile_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_test = add_feature_copies(test_features)
    pipeline = Pipeline(
        [
            ("preprocessor", make_preprocessor(weighted_train)),
            (
                "model",
                ExtraTreesRegressor(
                    n_estimators=N_ESTIMATORS,
                    min_samples_leaf=1,
                    max_features=1,
                    random_state=RANDOM_SEED,
                    n_jobs=1,
                ),
            ),
        ]
    )
    pipeline.fit(weighted_train, target)
    matrix = pipeline.named_steps["preprocessor"].transform(weighted_test)
    forest = pipeline.named_steps["model"]
    tree_predictions = np.column_stack(
        [tree.predict(matrix) for tree in forest.estimators_]
    )
    return np.clip(
        np.round(np.quantile(tree_predictions, TREE_QUANTILE, axis=1), 2),
        0.0,
        1.0,
    )


def empirical_rank_transform(
    train_values: np.ndarray,
    test_values: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Train 분포만으로 Train과 한 개씩 독립적인 Test 값을 변환."""
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(
        sorted_train, train_values, side="right"
    ) / len(train_values)
    test_rank = np.searchsorted(
        sorted_train, test_values, side="right"
    ) / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_neighbor_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    """각 2차원 피처쌍에서 가장 가까운 Train target을 분위수 집계."""
    train_rank = {}
    test_rank = {}
    for column in PAIR_COLUMNS:
        train_rank[column], test_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            test_features[column].to_numpy(dtype=float),
        )

    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(test_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(test_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_index = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_index])
    return np.quantile(
        np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1
    )


def validate_inputs(
    train: pd.DataFrame,
    test: pd.DataFrame,
    submission: pd.DataFrame,
) -> None:
    if set(train.columns) - {TARGET} != set(test.columns):
        raise ValueError("Train과 Test의 입력 컬럼이 다릅니다.")
    if list(submission.columns) != [ID_COLUMN, TARGET]:
        raise ValueError("제출 양식 컬럼이 올바르지 않습니다.")
    if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
        raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")
    if train[TARGET].isna().any():
        raise ValueError("Train 타깃에 결측값이 있습니다.")

In [2]:
from sklearn.model_selection import ShuffleSplit

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project")
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
validate_inputs(train, test, submission)
X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
X_test = add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

SEEDS = (42, 77, 2026)
N_SPLITS = 8
VALID_SIZE = 150
BASE_TREES = 1200
TREES_PER_SEED = 400
TREE_Q_GRID = (0.50, 0.52, 0.54)
PAIR_NAMES = list(combinations(PAIR_COLUMNS, 2))


def pair_prediction(train_features, valid_features, target):
    train_rank, valid_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], valid_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(float),
            valid_features[column].to_numpy(float),
        )
    target_values = target.to_numpy(float)
    output = []
    for first, second in PAIR_NAMES:
        distance = (
            np.abs(valid_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(valid_rank[second][:, None] - train_rank[second][None, :])
        )
        output.append(target_values[np.argmin(distance, axis=1)])
    return np.quantile(np.column_stack(output), 0.52, axis=1)


def fit_tree_matrix(train_features, valid_features, target, trees, seed):
    weighted_train = add_feature_copies(train_features)
    weighted_valid = add_feature_copies(valid_features)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    valid_matrix = preprocessor.transform(weighted_valid)
    forest = ExtraTreesRegressor(
        n_estimators=trees,
        min_samples_leaf=1,
        max_features=1,
        random_state=seed,
        n_jobs=-1,
    )
    forest.fit(train_matrix, target)
    return np.column_stack(
        [tree.predict(valid_matrix) for tree in forest.estimators_]
    )


def blend(tree_prediction, pair_prediction_value):
    return np.clip(
        np.round(0.85 * tree_prediction + 0.15 * pair_prediction_value, 2),
        0.0, 1.0,
    )


splitter = ShuffleSplit(
    n_splits=N_SPLITS,
    test_size=VALID_SIZE,
    random_state=29082026,
)
rows = []

for split_id, (train_idx, valid_idx) in enumerate(splitter.split(X), 1):
    fold_train, fold_valid = X.iloc[train_idx], X.iloc[valid_idx]
    fold_target = y.iloc[train_idx]
    valid_target = y.iloc[valid_idx]
    pair_pred = pair_prediction(fold_train, fold_valid, fold_target)

    baseline_matrix = fit_tree_matrix(
        fold_train, fold_valid, fold_target, BASE_TREES, 42
    )
    baseline = blend(np.quantile(baseline_matrix, 0.52, axis=1), pair_pred)
    baseline_mae = mean_absolute_error(valid_target, baseline)

    seed_matrices = [
        fit_tree_matrix(
            fold_train, fold_valid, fold_target,
            TREES_PER_SEED, seed,
        )
        for seed in SEEDS
    ]
    pooled = np.column_stack(seed_matrices)

    for tree_q in TREE_Q_GRID:
        pooled_tree = np.quantile(pooled, tree_q, axis=1)
        pooled_prediction = blend(pooled_tree, pair_pred)
        pooled_mae = mean_absolute_error(valid_target, pooled_prediction)
        rows.append({
            "split": split_id,
            "method": "pooled_quantile",
            "tree_q": tree_q,
            "baseline_MAE": baseline_mae,
            "candidate_MAE": pooled_mae,
            "improvement": baseline_mae - pooled_mae,
        })

        seed_quantiles = np.column_stack([
            np.quantile(matrix, tree_q, axis=1)
            for matrix in seed_matrices
        ])
        for method, tree_pred in {
            "mean_seed_quantile": seed_quantiles.mean(axis=1),
            "median_seed_quantile": np.median(seed_quantiles, axis=1),
        }.items():
            prediction = blend(tree_pred, pair_pred)
            score = mean_absolute_error(valid_target, prediction)
            rows.append({
                "split": split_id,
                "method": method,
                "tree_q": tree_q,
                "baseline_MAE": baseline_mae,
                "candidate_MAE": score,
                "improvement": baseline_mae - score,
            })
    print(f"V29 Near-Full split {split_id}/{N_SPLITS} complete")

score_frame = pd.DataFrame(rows)
summary_rows = []
for (method, tree_q), part in score_frame.groupby(["method", "tree_q"]):
    summary_rows.append({
        "method": method,
        "tree_q": tree_q,
        "audit_MAE": part.candidate_MAE.mean(),
        "audit_mean_improvement": part.improvement.mean(),
        "audit_worst_improvement": part.improvement.min(),
        "audit_wins": int((part.improvement > 0).sum()),
    })

summary = pd.DataFrame(summary_rows).sort_values(
    ["audit_wins", "audit_worst_improvement", "audit_MAE"],
    ascending=[False, False, True],
).reset_index(drop=True)
print("Baseline Near-Full MAE:", f"{score_frame.groupby('split').baseline_MAE.first().mean():.6f}")
display(summary)

eligible = summary[
    (summary.audit_wins >= 6)
    & (summary.audit_mean_improvement >= 0.0003)
    & (summary.audit_worst_improvement >= -0.0005)
]

if eligible.empty:
    print("V29 rejected: Seed Bagging 후보가 감사 기준을 통과하지 못했습니다.")
    print("제출 파일을 생성하지 않습니다.")
    v29_accepted = False
else:
    selected = eligible.iloc[0]
    v29_accepted = True
    print("V29 accepted")
    print(selected)
    final_pair = pair_prediction(X, X_test, y)
    final_matrices = [
        fit_tree_matrix(X, X_test, y, TREES_PER_SEED, seed)
        for seed in SEEDS
    ]
    tree_q = float(selected.tree_q)
    if selected.method == "pooled_quantile":
        final_tree = np.quantile(
            np.column_stack(final_matrices), tree_q, axis=1
        )
    else:
        seed_quantiles = np.column_stack([
            np.quantile(matrix, tree_q, axis=1)
            for matrix in final_matrices
        ])
        if selected.method == "mean_seed_quantile":
            final_tree = seed_quantiles.mean(axis=1)
        else:
            final_tree = np.median(seed_quantiles, axis=1)
    final_prediction = blend(final_tree, final_pair)
    final_submission = submission.copy()
    final_submission[TARGET] = final_prediction
    final_path = OUTPUT_DIR / "submit_v29_seed_bagging.csv"
    final_submission.to_csv(final_path, index=False, encoding="utf-8")
    print("Saved:", final_path)
    display(final_submission.head(10))



V29 Near-Full split 1/8 complete
V29 Near-Full split 2/8 complete
V29 Near-Full split 3/8 complete
V29 Near-Full split 4/8 complete
V29 Near-Full split 5/8 complete
V29 Near-Full split 6/8 complete
V29 Near-Full split 7/8 complete
V29 Near-Full split 8/8 complete
Baseline Near-Full MAE: 0.124175


,method,tree_q,audit_MAE,audit_mean_improvement,audit_worst_improvement,audit_wins
0,pooled_quantile,0.50,0.124333,-0.000158,-0.002000,4
1,pooled_quantile,0.54,0.124942,-0.000767,-0.004667,4
2,mean_seed_quantile,0.50,0.124358,-0.000183,-0.002133,3
3,pooled_quantile,0.52,0.124600,-0.000425,-0.003267,3
4,mean_seed_quantile,0.52,0.124408,-0.000233,-0.003533,3
5,mean_seed_quantile,0.54,0.124617,-0.000442,-0.004200,3
6,median_seed_quantile,0.50,0.124633,-0.000458,-0.003133,2
7,median_seed_quantile,0.52,0.124858,-0.000683,-0.003733,2
8,median_seed_quantile,0.54,0.125275,-0.001100,-0.004867,2


V29 rejected: Seed Bagging 후보가 감사 기준을 통과하지 못했습니다.
제출 파일을 생성하지 않습니다.
